# Teams & Ownership

Resolve git contributors to teams, compute code ownership, identify bus factor risks.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

## 1. Team Configuration

In [ ]:
try:
    from buildanalysis.teams import TeamConfig
    
    teams_yaml = PROJECT_ROOT / "teams.yaml"
    if teams_yaml.exists():
        print(f"Attempting to load teams configuration from {teams_yaml}...")
        try:
            team_config = TeamConfig.from_yaml(teams_yaml)
            print(f"\nTeam configuration loaded successfully.")
            print(f"Total teams: {len(team_config.teams)}")
            print(f"Total unaffiliated members: {len(team_config.unaffiliated)}")
            print(f"Total email mappings: {len(team_config.email_to_team)}")
            
            for team_name, members in team_config.teams.items():
                print(f"\n{team_name}: {len(members)} member(s)")
                for member in members:
                    print(f"  - {member.name}: {len(member.emails)} email(s)")
        except Exception as e:
            print(f"Error loading team config: {e}")
            print("Proceeding without team configuration.")
            team_config = None
    else:
        print(f"teams.yaml not found at {teams_yaml}")
        print("Proceeding without team configuration.")
        team_config = None
except ImportError:
    print("TeamConfig import failed. Skipping team configuration.")
    team_config = None

## 2. Git Contributor Resolution

In [ ]:
if not ds.has_file("git_commit_log"):
    print("git_commit_log.parquet not available. Skipping contributor resolution.")
else:
    git_log = ds.git_commit_log
    
    total_contributors = git_log["contributor"].nunique()
    total_commits = len(git_log)
    
    print(f"\nGIT CONTRIBUTOR ANALYSIS")
    print(f"=" * 70)
    print(f"Total commits: {total_commits:,}")
    print(f"Unique contributors: {total_contributors}")
    
    # Contributor summary
    contrib_summary = git_log["contributor"].value_counts().reset_index()
    contrib_summary.columns = ["Contributor", "Commit Count"]
    contrib_summary["Commit %"] = (100 * contrib_summary["Commit Count"] / total_commits).round(1)
    
    print(f"\nTop 10 contributors by commit count:")
    print(contrib_summary.head(10).to_string(index=False))
    
    print(f"\nContributor concentration (Herfindahl-Hirschman Index):")
    market_shares = contrib_summary["Commit Count"] / total_commits
    hhi = (market_shares ** 2).sum()
    print(f"HHI: {hhi:.4f}")
    if hhi > 0.25:
        print("  -> High concentration (top contributors dominate)")
    elif hhi > 0.10:
        print("  -> Moderate concentration")
    else:
        print("  -> Low concentration (distributed contribution)")

## 3. Target Ownership

In [ ]:
if not ds.has_file("contributor_target_commits"):
    print("contributor_target_commits.parquet not available. Skipping target ownership analysis.")
else:
    ctc = ds.contributor_target_commits
    tm = ds.target_metrics
    
    print(f"\nTARGET OWNERSHIP ANALYSIS")
    print(f"=" * 70)
    
    # Compute ownership concentration per target
    target_owner_summary = []
    
    for target in ctc["cmake_target"].unique():
        target_data = ctc[ctc["cmake_target"] == target]
        total_commits = target_data["commit_count"].sum()
        unique_contributors = len(target_data)
        
        # HHI for this target
        market_shares = target_data["commit_count"] / total_commits
        hhi = (market_shares ** 2).sum()
        
        # Bus factor (number of people with majority of commits)
        sorted_commits = target_data.sort_values("commit_count", ascending=False)
        cumsum = sorted_commits["commit_count"].cumsum()
        bus_factor = (cumsum <= 0.5 * total_commits).sum() + 1
        
        target_owner_summary.append({
            "target": target,
            "contributors": unique_contributors,
            "total_commits": total_commits,
            "hhi": hhi,
            "bus_factor": bus_factor
        })
    
    ownership_df = pd.DataFrame(target_owner_summary)
    
    # Report HHI distribution
    print(f"\nOwnership concentration (HHI):")
    print(f"  Mean: {ownership_df['hhi'].mean():.4f}")
    print(f"  Median: {ownership_df['hhi'].median():.4f}")
    print(f"  Min: {ownership_df['hhi'].min():.4f}")
    print(f"  Max: {ownership_df['hhi'].max():.4f}")
    
    # Bus factor analysis
    print(f"\nBus factor (contributors needed for 50% of commits):")
    bf_counts = ownership_df["bus_factor"].value_counts().sort_index()
    for bf, count in bf_counts.items():
        print(f"  Factor {bf}: {count:,} target(s)")
    
    # High risk targets (bus factor = 1)
    single_owner = ownership_df[ownership_df["bus_factor"] == 1]
    if len(single_owner) > 0:
        print(f"\nWARNING: {len(single_owner):,} target(s) with single dominant contributor (bus factor = 1)")
        print("Top 10 high-risk targets:")
        high_risk = single_owner.sort_values("total_commits", ascending=False).head(10)
        for idx, row in high_risk.iterrows():
            print(f"  - {row['target']}: {row['contributors']} contributors, {row['total_commits']} commits")

## 4. Team Build Time Breakdown

In [ ]:
if team_config is None or not ds.has_file("contributor_target_commits") or not ds.has_file("target_metrics"):
    print("Team configuration or required data not available. Skipping team build time breakdown.")
else:
    ctc = ds.contributor_target_commits
    tm = ds.target_metrics
    
    print(f"\nTEAM BUILD TIME BREAKDOWN")
    print(f"=" * 70)
    
    # Assign contributors to teams
    ctc_copy = ctc.copy()
    ctc_copy["team"] = ctc_copy["contributor"].map(
        lambda x: team_config.email_to_team.get(x, "__unaffiliated__")
    )
    
    # Group targets by primary owning team (team with most commits)
    team_commits = ctc_copy.groupby(["cmake_target", "team"])["commit_count"].sum().reset_index()
    primary_team = team_commits.sort_values("commit_count", ascending=False).drop_duplicates("cmake_target")
    
    # Merge with target metrics for build time
    team_build = primary_team.merge(tm[["cmake_target", "total_build_time_ms"]], on="cmake_target", how="left")
    team_build["total_build_time_ms"] = team_build["total_build_time_ms"].fillna(0)
    
    # Sum by team
    team_summary = team_build.groupby("team")["total_build_time_ms"].sum().reset_index()
    team_summary.columns = ["Team", "Build Time (ms)"]
    team_summary["Build Time (sec)"] = (team_summary["Build Time (ms)"] / 1000).round(2)
    team_summary = team_summary.sort_values("Build Time (ms)", ascending=False)
    
    print("\nBuild time contribution by owning team:")
    print(team_summary[["Team", "Build Time (sec)"]].to_string(index=False))
    
    # Visualize
    if len(team_summary) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(team_summary["Team"], team_summary["Build Time (sec)"])
        ax.set_xlabel("Build Time (seconds)")
        ax.set_title("Build Time Contribution by Team")
        plt.tight_layout()
        plt.show()

## 5. Team Coupling

In [ ]:
if team_config is None or not ds.has_file("edge_list") or not ds.has_file("contributor_target_commits"):
    print("Team configuration or required data not available. Skipping team coupling analysis.")
else:
    edge_list = ds.edge_list
    ctc = ds.contributor_target_commits
    
    print(f"\nTEAM COUPLING ANALYSIS")
    print(f"=" * 70)
    
    # Map targets to teams
    ctc_copy = ctc.copy()
    ctc_copy["team"] = ctc_copy["contributor"].map(
        lambda x: team_config.email_to_team.get(x, "__unaffiliated__")
    )
    target_to_team = ctc_copy.drop_duplicates("cmake_target").set_index("cmake_target")["team"].to_dict()
    
    # Add team info to edge list
    edge_list_copy = edge_list.copy()
    edge_list_copy["source_team"] = edge_list_copy["source_target"].map(target_to_team)
    edge_list_copy["dest_team"] = edge_list_copy["dest_target"].map(target_to_team)
    
    # Count edges between teams
    inter_team_edges = edge_list_copy[
        (edge_list_copy["source_team"].notna()) & 
        (edge_list_copy["dest_team"].notna()) &
        (edge_list_copy["source_team"] != edge_list_copy["dest_team"])
    ]
    
    coupling_summary = inter_team_edges.groupby(["source_team", "dest_team"]).size().reset_index(name="edge_count")
    
    print(f"\nInter-team dependencies (top 20):")
    top_couplings = coupling_summary.sort_values("edge_count", ascending=False).head(20)
    for idx, row in top_couplings.iterrows():
        print(f"  {row['source_team']:30} -> {row['dest_team']:30} : {row['edge_count']:3} edges")
    
    # Build coupling matrix
    teams_list = sorted(set(target_to_team.values()))
    coupling_matrix = pd.DataFrame(0, index=teams_list, columns=teams_list)
    
    for idx, row in coupling_summary.iterrows():
        if row['source_team'] in coupling_matrix.index and row['dest_team'] in coupling_matrix.columns:
            coupling_matrix.loc[row['source_team'], row['dest_team']] += row['edge_count']
    
    print(f"\nCoupling matrix (team × team dependency count):")
    print(coupling_matrix)
    
    # Visualize
    if len(teams_list) > 1 and len(teams_list) <= 10:
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(coupling_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
        ax.set_title("Team Dependency Coupling Matrix")
        plt.tight_layout()
        plt.show()

## 6. Save Intermediates

In [ ]:
# Prepare target_ownership intermediate if we have contributor data
if 'ownership_df' in locals() and ds.has_file("target_metrics"):
    tm = ds.target_metrics
    target_ownership = ownership_df.merge(tm[["cmake_target", "target_type", "total_build_time_ms"]], 
                                           left_on="target", right_on="cmake_target", how="left")
    
    # Save
    path = ds.save_intermediate("target_ownership", target_ownership[["target", "contributors", "hhi", "bus_factor"]])
    print(f"Saved target_ownership to {path}")

print("\nNotebook completed successfully.")